# 06 · Una base analítica, sin multiplicar producción

**Entradas:** los cinco Parquet limpios y el complemento WebFlor limpio. No se leen Excel ni se conecta a SQL.
**Salida oficial única:** `Datos_analiticos_proyecto/base_analitica.parquet`.

**Grano:** una semana con producción observada por finca × producto × color × variedad. Alcance: Gaitana desde 2021. No convertir semanas ausentes en cero.

La producción es la tabla izquierda de cada unión. Las emisiones H1–H5 del ingeniero se representan en columnas; el histórico completo de emisiones sigue en su base limpia. El archivo WebFlor no se convierte artificialmente en corridas conocidas al origen.

Esta tabla es principalmente **retrospectiva**. Incluye clima contemporáneo y un potencial descriptivo: el notebook 08 separa qué columnas podrían usarse predictivamente. Tener una columna no autoriza usarla para pronosticar el pasado.

## 1. Imports y rutas simples

In [1]:
from pathlib import Path
import sys
import re
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Datos").is_dir() and (p / "Notebooks").is_dir())


SALIDA = ROOT / "Datos_analiticos_proyecto"
SALIDA.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 35)
print("Fuentes originales:", ROOT / "Datos")
print("Salida oficial:", SALIDA)

KEY = ["finca","producto","color","variedad"]
SEM = KEY+["semana_inicio"]
archivos = ["produccion_real_limpio","clima_limpio","planos_siembra_limpio",
            "curvas_variedad_limpio","estimados_semanales_limpio","proyecciones_webflor_limpio","coeficientes_webflor_limpio"]
for nombre in archivos:
    assert (SALIDA/(nombre+".parquet")).is_file(),f"Ejecutar limpieza de {nombre}"
display(pd.DataFrame({"fuente":[str(SALIDA/(n+".parquet")) for n in archivos]}))


Fuentes originales: C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos
Salida oficial: C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos_analiticos_proyecto


,fuente
0,C:\Materias y Cosas Técnicas Uniandes\Proyecto...
1,C:\Materias y Cosas Técnicas Uniandes\Proyecto...
2,C:\Materias y Cosas Técnicas Uniandes\Proyecto...
3,C:\Materias y Cosas Técnicas Uniandes\Proyecto...
4,C:\Materias y Cosas Técnicas Uniandes\Proyecto...
5,C:\Materias y Cosas Técnicas Uniandes\Proyecto...
6,C:\Materias y Cosas Técnicas Uniandes\Proyecto...


## Decisiones sobre todas las fuentes, antes de unir

Revisamos las siete bases: una fuente auxiliar no se fuerza a la tabla principal. Las coberturas son observadas en archivos locales, no una certificación de disponibilidad histórica. No se crean catálogos externos ni otras bases.


In [2]:
import pyarrow.parquet as pq
decisiones = [
    ["produccion_real_limpio","Movimiento por libro/hoja/fila","Tallos reportados","PRINCIPAL",
     "Target; sumar movimientos, nunca duplicar con cortes SQL.","fecha_corte"],
    ["estimados_semanales_limpio","Solicitud + archivo/fila_cache + objetivo","Tallos ESTI. EXPORTABLE","BENCHMARK",
     "Conservar exactos/posteriores separados; emisión y equivalencia de unidades pendientes.","semana_inicio"],
    ["planos_siembra_limpio","Línea de plano semanal; edición elegida","Plantas/área/edad","PRINCIPAL",
     "Agregar antes del join; validar edad contra AL1, no contra una fecha distinta.","semana_inicio"],
    ["curvas_variedad_limpio","Observación de piloto/siembra/semana","Flores por planta","POTENCIAL",
     "Proxy descriptivo; la versión predictiva debe usar sólo ediciones previas al origen.","semana_inicio"],
    ["clima_limpio","Medición por finca/instante","Unidad fuente por confirmar","PRINCIPAL",
     "Medias diarias y semanales; rezagar. No sumar radiación/lluvia sin unidad/frecuencia confirmadas.","fecha"],
    ["proyecciones_webflor_limpio","Estado/registro SQL por siembra/objetivo","Tallos proyectados","AUXILIAR + ESTADO",
     "No sumar actual/archivo/diario. Archivo de erradicación no certifica origen de forecast.","semana_inicio"],
    ["coeficientes_webflor_limpio","IdCurvaCoeficiente; curva/edad","Coeficiente por planta o área","AUXILIAR",
     "Los planos no traen IdCurvaCiclo y no hay vigencia histórica certificada: no elegir una curva arbitraria.",None],
]
catalogo = pd.DataFrame(decisiones,columns=["fuente","grano","unidad","decision","razon","campo_temporal"])
filas,desde,hasta = [],[],[]
for fila in catalogo.itertuples():
    ruta = SALIDA/(fila.fuente+".parquet")
    filas.append(pq.read_metadata(ruta).num_rows)
    fechas = pd.read_parquet(ruta,columns=[fila.campo_temporal])[fila.campo_temporal] if pd.notna(fila.campo_temporal) else pd.Series([],dtype="datetime64[ns]")
    desde.append(fechas.min());hasta.append(fechas.max())
catalogo["filas"],catalogo["desde"],catalogo["hasta"] = filas,desde,hasta
display(catalogo)
print("Se revisan las siete fuentes limpias. Esta tabla de decisiones queda dentro del notebook.")


,fuente,grano,unidad,decision,razon,campo_temporal,filas,desde,hasta
0,produccion_real_limpio,Movimiento por libro/hoja/fila,Tallos reportados,PRINCIPAL,"Target; sumar movimientos, nunca duplicar con ...",fecha_corte,6389201,2021-01-04,2026-09-05
1,estimados_semanales_limpio,Solicitud + archivo/fila_cache + objetivo,Tallos ESTI. EXPORTABLE,BENCHMARK,Conservar exactos/posteriores separados; emisi...,semana_inicio,176140,2023-01-02,2026-10-05
2,planos_siembra_limpio,Línea de plano semanal; edición elegida,Plantas/área/edad,PRINCIPAL,Agregar antes del join; validar edad contra AL...,semana_inicio,3252152,2021-01-04,2026-08-10
3,curvas_variedad_limpio,Observación de piloto/siembra/semana,Flores por planta,POTENCIAL,Proxy descriptivo; la versión predictiva debe ...,semana_inicio,549870,2012-12-31,2026-07-13
4,clima_limpio,Medición por finca/instante,Unidad fuente por confirmar,PRINCIPAL,Medias diarias y semanales; rezagar. No sumar ...,fecha,187134,2021-01-01,2026-08-11
5,proyecciones_webflor_limpio,Estado/registro SQL por siembra/objetivo,Tallos proyectados,AUXILIAR + ESTADO,No sumar actual/archivo/diario. Archivo de err...,semana_inicio,1686034,2020-10-12,2026-12-21
6,coeficientes_webflor_limpio,IdCurvaCoeficiente; curva/edad,Coeficiente por planta o área,AUXILIAR,Los planos no traen IdCurvaCiclo y no hay vige...,NaN,128567,NaT,NaT


Se revisan las siete fuentes limpias. Esta tabla de decisiones queda dentro del notebook.


## 2. Producción: agregar una sola vez

Primero se filtra finca exacta GAITANA. GAITANA PPTO es distinta y no se incorpora por contener la palabra Gaitana. No se suman los cortes SQL ni la clasificación. Agrupar movimientos legítimos no equivale a desduplicarlos.

In [3]:
real = pd.read_parquet(SALIDA/"produccion_real_limpio.parquet",columns=KEY+["semana_inicio","fecha_corte","tallos","llave_completa"])
real = real.loc[real.finca.eq("GAITANA") & real.semana_inicio.ge("2021-01-01")].copy()
assert real.llave_completa.all()
tallos_fuente = real.tallos.sum()
base = real.groupby(SEM,dropna=False).agg(
    tallos_reales=("tallos","sum"),registros_corte=("tallos","size"),
    dias_con_reporte=("fecha_corte","nunique")).reset_index()
assert not base.duplicated(SEM).any()
assert base.tallos_reales.sum()==tallos_fuente
referencia_real = base[SEM+["tallos_reales"]].copy()  # sólo memoria, nunca otra exportación
base["unidad"] = "tallos reportados"
base["produccion_observada"] = True
iso = base.semana_inicio.dt.isocalendar()
base["anio"],base["semana"] = iso.year,iso.week
print("Movimientos:",len(real),"Filas semanales:",len(base),"Tallos:",int(tallos_fuente))
display(base.groupby("producto").agg(filas=("tallos_reales","size"),tallos=("tallos_reales","sum"),
                                   variedades=("variedad","nunique")))
display(base.head())
del real
reconciliaciones = []


Movimientos: 4654552 Filas semanales: 44053 Tallos: 327991200


,filas,tallos,variedades
producto,,,
ACHILLEA,209,14149,4
AMAZON,256,653257,2
BARBATUS,1630,2193346,43
BRASSICA,7,982,2
CAMPANULA,194,89657,3
CARNATION,18849,60581236,278
FIORINO,8,2776,1
GREEN BALL,580,5839878,9
LEPIDIUM,56,36291,1


,finca,producto,color,variedad,semana_inicio,tallos_reales,registros_corte,dias_con_reporte,unidad,produccion_observada,anio,semana
0,GAITANA,ACHILLEA,LIGHT PINK,SASSY SUMMER TAFFY,2023-07-17,60,1,1,tallos reportados,True,2023,29
1,GAITANA,ACHILLEA,LIGHT PINK,SASSY SUMMER TAFFY,2023-07-24,185,4,3,tallos reportados,True,2023,30
2,GAITANA,ACHILLEA,LIGHT PINK,SASSY SUMMER TAFFY,2023-07-31,180,3,3,tallos reportados,True,2023,31
3,GAITANA,ACHILLEA,LIGHT PINK,SASSY SUMMER TAFFY,2023-08-07,120,2,2,tallos reportados,True,2023,32
4,GAITANA,ACHILLEA,LIGHT PINK,SASSY SUMMER TAFFY,2023-08-21,60,1,1,tallos reportados,True,2023,34


## 3. Clima: medición → día → semana → producción

Una fila de clima puede corresponder a muchas variedades, por lo que se agrega a finca/semana antes del N:1. Primero se promedian mediciones de cada día y después días de la semana: así un día con frecuencia horaria no pesa la mitad de uno con frecuencia de media hora. Se muestra cobertura; no se llena clima faltante.

La fecha/hora de pared reportada se conserva; los sufijos de zona horaria detectados requieren validación con la estación. No se da por certificada su latencia.

In [4]:
clima = pd.read_parquet(SALIDA/"clima_limpio.parquet")
clima = clima.loc[clima.finca.eq("GAITANA") & clima.medicion_valida].copy()
diario = clima.groupby(["finca","fecha","semana_inicio"]).agg(
    temperatura=("temperatura","mean"),humedad=("humedad","mean"),radiacion=("radiacion","mean"),
    temperatura_min=("temperatura_min","min"),temperatura_max=("temperatura_max","max"),
    mediciones=("instante","size"),hora_con_zona=("hora_con_zona","max")).reset_index()
clima_sem = diario.groupby(["finca","semana_inicio"]).agg(
    temperatura_semana=("temperatura","mean"),humedad_semana=("humedad","mean"),
    radiacion_semana=("radiacion","mean"),temperatura_min_semana=("temperatura_min","min"),
    temperatura_max_semana=("temperatura_max","max"),dias_clima=("fecha","nunique"),
    mediciones_clima=("mediciones","sum"),horario_clima_por_validar=("hora_con_zona","max")).reset_index()
antes = len(base)
base = base.merge(clima_sem,on=["finca","semana_inicio"],how="left",validate="many_to_one",indicator=True)
reconciliaciones.append({"union":"clima","cardinalidad":"N:1","antes":antes,"despues":len(base),
 "sin_pareja":int(base._merge.eq("left_only").sum()),"tallos":int(base.tallos_reales.sum())})
base = base.drop(columns="_merge")
assert len(base)==antes and base.tallos_reales.sum()==tallos_fuente
display(clima_sem.tail())
del clima, diario


,finca,semana_inicio,temperatura_semana,humedad_semana,radiacion_semana,temperatura_min_semana,temperatura_max_semana,dias_clima,mediciones_clima,horario_clima_por_validar
286,GAITANA,2026-06-29,14.381548,85.413690,155.187500,7.6,21.9,7,336,False
287,GAITANA,2026-07-06,14.685714,83.169643,154.958333,5.1,21.2,7,336,False
288,GAITANA,2026-07-13,14.810714,80.752976,165.738095,6.1,21.7,7,336,False
289,GAITANA,2026-07-20,14.289583,84.089286,167.639881,3.9,21.6,7,336,False
290,GAITANA,2026-07-27,11.737595,86.712121,165.903409,5.4,20.8,2,70,False


## 4. Siembras + curva empírica: potencial descriptivo, no WebFlor

La selección de edición del plano está declarada en su limpieza. No se mezclan dos planos de la misma semana. Se usa coincidencia semanal exacta; no se prolonga indefinidamente una observación anterior.

El color del piloto suele estar en otra nomenclatura. **No traducimos por parecido**: sólo se asigna color comercial si producto/variedad identifica una única combinación en producción. Las ambigüedades quedan sin curva vinculada.

Hipótesis: plantas y edad describen potencial. Se calcula en cada línea admisible del plano `plantas × mediana(flores/planta/semana de la edad)`, y después se suma. Se exige soporte ≥10 observaciones por curva y cobertura ≥90% de plantas. No se extrapola la parte no cubierta.

La limpieza 04 ahora valida la edad reportada contra la fecha AL1 usada por la fórmula original, comprobada en los 293 libros con Datos. La edad al lunes sigue calculándose desde fecha_siembra: una diferencia con la edad de otra fecha no invalida automáticamente la siembra. La fecha AL1 tampoco se interpreta como publicación del plano.

**Limitación:** esta mediana usa todas las observaciones piloto admitidas; es descriptiva y está prohibida como feature histórica. El supuesto flores≈tallos y la representatividad de pilotos requieren validación. No es una reproducción de la fórmula WebFlor.

In [5]:
planos = pd.read_parquet(SALIDA/"planos_siembra_limpio.parquet")
planos = planos.loc[planos.finca.eq("GAITANA") & planos.edicion_elegida & planos[SEM].notna().all(axis=1)].copy()
curvas = pd.read_parquet(SALIDA/"curvas_variedad_limpio.parquet")
curvas = curvas.loc[curvas.observacion_valida].copy()
dim = base[["producto","color","variedad"]].drop_duplicates()
dim_unica = dim.loc[~dim.duplicated(["producto","variedad"],keep=False)]
curvas = curvas.merge(dim_unica,on=["producto","variedad"],how="left",validate="many_to_one")
print("Eventos piloto sin color comercial unívoco:",int(curvas.color.isna().sum()))
curva = curvas.loc[curvas.color.notna()].groupby(["producto","color","variedad","edad_semanas"]).agg(
    rendimiento_mediano=("flores_por_planta","median"),soporte_curva=("flores_por_planta","size")).reset_index()
assert not curva.duplicated(["producto","color","variedad","edad_semanas"]).any()
planos["edad_semanas"] = planos.edad_calculada
n_planos = len(planos)
planos = planos.merge(curva,on=["producto","color","variedad","edad_semanas"],how="left",validate="many_to_one")
assert len(planos)==n_planos
admisible = planos.fila_utilizable & ~planos.edad_inconsistente & planos.edad_semanas.ge(0)
cubierta = admisible & planos.soporte_curva.ge(10)
planos["plantas_cubiertas"] = planos.plantas.where(cubierta,0)
planos["potencial_linea"] = (planos.plantas*planos.rendimiento_mediano).where(cubierta)
planos["plantas_edad_conocida"] = planos.plantas.where(admisible,0)
planos["plantas_por_edad"] = (planos.plantas*planos.edad_semanas).where(admisible,0)
planos["plantas_ambiguas"] = planos.plantas.where(planos.segmento_ambiguo,0)
planos["cama_identidad"] = planos.invernadero.astype("string")+"|"+planos.cama.astype("string")
siembra_sem = planos.groupby(SEM).agg(
    plantas_plano=("plantas",lambda s:s.sum(min_count=1)),
    area_plano=("area",lambda s:s.sum(min_count=1)),
    camas_plano=("cama_identidad","nunique"),filas_plano=("cama","size"),
    plantas_cubiertas=("plantas_cubiertas","sum"),plantas_edad_conocida=("plantas_edad_conocida","sum"),
    plantas_por_edad=("plantas_por_edad","sum"),plantas_ambiguas=("plantas_ambiguas","sum"),
    potencial_parcial=("potencial_linea",lambda s:s.sum(min_count=1)),
    archivo_plano=("archivo_origen","first"),
    fecha_referencia_edad_plano=("fecha_referencia_edad","max")).reset_index()
siembra_sem["edad_media_plano"] = siembra_sem.plantas_por_edad/siembra_sem.plantas_edad_conocida.replace(0,np.nan)
siembra_sem["cobertura_curva"] = siembra_sem.plantas_cubiertas/siembra_sem.plantas_plano.replace(0,np.nan)
siembra_sem["potencial_descriptivo"] = siembra_sem.potencial_parcial.where(
    siembra_sem.cobertura_curva.ge(.9) & siembra_sem.plantas_ambiguas.eq(0))
antes = len(base)
base = base.merge(siembra_sem,on=SEM,how="left",validate="one_to_one",indicator=True)
reconciliaciones.append({"union":"planos_curvas","cardinalidad":"1:1","antes":antes,"despues":len(base),
 "sin_pareja":int(base._merge.eq("left_only").sum()),"tallos":int(base.tallos_reales.sum())})
base = base.drop(columns="_merge")
assert len(base)==antes and base.tallos_reales.sum()==tallos_fuente
base["desviacion_potencial_descriptiva"] = base.tallos_reales-base.potencial_descriptivo
display(base[SEM+["plantas_plano","edad_media_plano","cobertura_curva","potencial_descriptivo"]].dropna(subset="potencial_descriptivo").head())
del planos, curvas, curva


Eventos piloto sin color comercial unívoco: 101605


,finca,producto,color,variedad,semana_inicio,plantas_plano,edad_media_plano,cobertura_curva,potencial_descriptivo
631,GAITANA,BARBATUS,BICOLOR PURPLE,ONYX,2023-08-07,3970.0,13.518892,1.0,784.50
632,GAITANA,BARBATUS,BICOLOR PURPLE,ONYX,2023-08-14,3970.0,14.518892,1.0,930.54
633,GAITANA,BARBATUS,BICOLOR PURPLE,ONYX,2023-08-21,3970.0,15.518892,1.0,1014.00
634,GAITANA,BARBATUS,BICOLOR PURPLE,ONYX,2023-08-28,3970.0,16.518892,1.0,1045.64
635,GAITANA,BARBATUS,BICOLOR PURPLE,ONYX,2023-09-04,1970.0,15.507614,1.0,691.28


## 5. Benchmark del ingeniero: misma recuperación que el original

La limpieza 03 ya aplica exacta o siguiente entrega del mismo año, conserva el origen solicitado y el archivo realmente usado y construye H1–H5. Aquí se agregan sus líneas por llave comercial/horizonte, como en `03_benchmark_estimado_ingeniero.ipynb`, sin sumar distintos archivos para la misma solicitud.

Primera unión: finca/producto/color/variedad/semana. Segunda: **sólo los pendientes**, por finca/producto/variedad/semana, con unicidad 1:1 comprobada, igual que la validación del benchmark 2026. Si hay varios colores candidatos no se escoge el primero ni se multiplica producción: quedan pendientes.

Se conserva el color de producción como dimensión oficial del panel y `color_estimado_ingeniero_h*` permite ver el color original del estimado; `metodo_union_ingeniero_h*` distingue coincidencia y recuperación por variedad. Es una correspondencia analítica, no una equivalencia taxonómica certificada. El benchmark original priorizaba el color del estimado; mantener ambos aquí evita cambiar retroactivamente el color de producción.

`archivo_posterior_ingeniero_h*` impide confundir una recuperación retrospectiva con disponibilidad al origen. Las cantidades sin producir y objetivos futuros siguen en el limpio, no se inventan ceros reales.


In [6]:
estimados = pd.read_parquet(SALIDA/"estimados_semanales_limpio.parquet")
e0 = estimados.loc[estimados.finca.eq("GAITANA") & estimados.en_h1_h5 & ~estimados.es_ajustado].copy()
e0 = e0.loc[e0[SEM+["origen_solicitado","horizonte"]].notna().all(axis=1)]
llave_estimado = SEM+["horizonte"]
assert e0.groupby(llave_estimado).archivo_origen.nunique().le(1).all(),"No sumar entregas diferentes"
e = e0.groupby(llave_estimado).agg(
    estimado=("estimado",lambda s:s.sum(min_count=1)),
    archivos=("archivo_origen","nunique"),archivo=("archivo_origen","first"),
    origen_solicitado=("origen_solicitado","first"),origen_nominal=("origen_nominal","first"),
    archivo_posterior=("archivo_es_posterior","max"),desfase_archivo=("diferencia_semanas_archivo","max"),
    registros_fuente=("estimado","size"),cantidades_presentes=("estimado","count")).reset_index()
e["conflicto_emision"] = False  # Un archivo principal elegido; no mezclar versiones.
e["cantidad_incompleta"] = e.cantidades_presentes.ne(e.registros_fuente)
llave_variedad = ["finca","producto","variedad","semana_inicio"]
control_benchmark = []
for h in range(1,6):
    ref = e.loc[e.horizonte.eq(h)].copy()
    ref["color_estimado"] = ref.color
    estricta = ref.merge(referencia_real[SEM],on=SEM,how="left",indicator=True,validate="one_to_one")
    coincide = estricta.loc[estricta._merge.eq("both")].drop(columns="_merge").copy()
    coincide["metodo_union"] = "coincide"
    est_pendiente = estricta.loc[estricta._merge.eq("left_only")].drop(columns="_merge")
    prod_pendiente = referencia_real.merge(coincide[SEM],on=SEM,how="left",indicator=True,validate="one_to_one")
    prod_pendiente = prod_pendiente.loc[prod_pendiente._merge.eq("left_only"),SEM]
    est_unico = est_pendiente.loc[~est_pendiente.duplicated(llave_variedad,keep=False)]
    prod_unico = prod_pendiente.loc[~prod_pendiente.duplicated(llave_variedad,keep=False)]
    recuperado = est_unico.drop(columns="color").merge(prod_unico,on=llave_variedad,how="inner",validate="one_to_one")
    recuperado["metodo_union"] = "unido_variedad"
    ref_final = pd.concat([coincide,recuperado],ignore_index=True)
    assert not ref_final.duplicated(SEM).any()
    control_benchmark.append({"horizonte":h,"referencias":len(ref),"coincide":len(coincide),
        "recuperadas_variedad":len(recuperado),"estimados_sin_pareja":len(ref)-len(ref_final),
        "produccion_sin_pareja":len(base)-len(ref_final),
        "parejas_archivo_posterior":int(ref_final.archivo_posterior.sum())})
    columnas = SEM+["estimado","origen_solicitado","origen_nominal","archivos","archivo",
                   "archivo_posterior","desfase_archivo","metodo_union","color_estimado","conflicto_emision","cantidad_incompleta"]
    nombres = {"estimado":f"estimado_ingeniero_h{h}","origen_solicitado":f"origen_ingeniero_h{h}",
       "origen_nominal":f"origen_archivo_ingeniero_h{h}","archivos":f"archivos_ingeniero_h{h}",
       "archivo":f"archivo_ingeniero_h{h}","archivo_posterior":f"archivo_posterior_ingeniero_h{h}",
       "desfase_archivo":f"desfase_archivo_ingeniero_h{h}","metodo_union":f"metodo_union_ingeniero_h{h}",
       "color_estimado":f"color_estimado_ingeniero_h{h}","conflicto_emision":f"conflicto_ingeniero_h{h}",
       "cantidad_incompleta":f"cantidad_incompleta_ingeniero_h{h}"}
    antes = len(base)
    base = base.merge(ref_final[columnas].rename(columns=nombres),on=SEM,how="left",validate="one_to_one",indicator=True)
    reconciliaciones.append({"union":f"ingeniero_h{h}","cardinalidad":"1:1","antes":antes,"despues":len(base),
      "sin_pareja":int(base._merge.eq("left_only").sum()),"tallos":int(base.tallos_reales.sum())})
    base = base.drop(columns="_merge")
    assert len(base)==antes and base.tallos_reales.sum()==tallos_fuente
    cantidad_completa = ~base[f"cantidad_incompleta_ingeniero_h{h}"].fillna(True).astype(bool)
    base[f"desviacion_ingeniero_h{h}"] = (base.tallos_reales-base[f"estimado_ingeniero_h{h}"]).where(cantidad_completa)
    base[f"desviacion_pct_ingeniero_h{h}"] = base[f"desviacion_ingeniero_h{h}"]/base[f"estimado_ingeniero_h{h}"].where(base[f"estimado_ingeniero_h{h}"].ge(100))
base["equivalencia_exportable_confirmada"] = False
base["fecha_emision_ingeniero_confirmada"] = False
display(pd.DataFrame(control_benchmark))
display(base.loc[base.semana_inicio.eq(pd.Timestamp.fromisocalendar(2026,2,1)),SEM+["estimado_ingeniero_h1","origen_ingeniero_h1","origen_archivo_ingeniero_h1","archivo_posterior_ingeniero_h1","metodo_union_ingeniero_h1"]].head(15))
del estimados, e0


,horizonte,referencias,coincide,recuperadas_variedad,estimados_sin_pareja,produccion_sin_pareja,parejas_archivo_posterior
0,1,24399,20611,1247,2541,22195,1397
1,2,24802,20786,1259,2757,22008,1760
2,3,24795,20616,1249,2930,22188,1757
3,4,24786,20458,1240,3088,22355,1748
4,5,24748,20264,1228,3256,22561,1745


,finca,producto,color,variedad,semana_inicio,estimado_ingeniero_h1,origen_ingeniero_h1,origen_archivo_ingeniero_h1,archivo_posterior_ingeniero_h1,metodo_union_ingeniero_h1
52,GAITANA,ACHILLEA,LIGHT PINK,SASSY SUMMER TAFFY,2026-01-05,40.0,2026-01-05,2026-01-12,True,coincide
128,GAITANA,ACHILLEA,RED,SASSY SUMMER SANGRIA,2026-01-05,30.0,2026-01-05,2026-01-12,True,coincide
194,GAITANA,ACHILLEA,YELLOW,SASSY SUMMER LEMON,2026-01-05,80.0,2026-01-05,2026-01-12,True,coincide
430,GAITANA,AMAZON,PURPLE,NEON PURPLE,2026-01-05,2850.0,2026-01-05,2026-01-12,True,coincide
835,GAITANA,BARBATUS,DARK PINK,TSAR,2026-01-05,NaN,NaT,NaT,NaN,NaN
916,GAITANA,BARBATUS,HOT PINK,WONDER,2026-01-05,1490.0,2026-01-05,2026-01-12,True,coincide
1171,GAITANA,BARBATUS,LAVENDER,MELIAN,2026-01-05,1990.0,2026-01-05,2026-01-12,True,coincide
1408,GAITANA,BARBATUS,LIGHT PINK,NOBLE,2026-01-05,1680.0,2026-01-05,2026-01-12,True,coincide
1473,GAITANA,BARBATUS,PURPLE,CALIPHA,2026-01-05,NaN,NaT,NaT,NaN,NaN
1723,GAITANA,BARBATUS,RED,AMRAS,2026-01-05,2050.0,2026-01-05,2026-01-12,True,coincide


## 6. WebFlor: referencia actual descargada, no error histórico certificado

Se agrega únicamente el estado ACTUAL de Gaitana por semana/dimensiones. No se suma con el archivo de erradicaciones. Se comprueba unicidad física antes de agregar y se muestra cobertura. El archivo completo y los coeficientes quedan como las dos bases auxiliares justificadas, sin forzarlos a una fila semanal.

Que la proyección permanezca varias semanas sin recalcularse no es un problema en sí. Falta demostrar cuándo estaba disponible y vigente para validar una corrección histórica. Por eso esta columna no entra a features del notebook 08.

In [7]:
wf = pd.read_parquet(SALIDA/"proyecciones_webflor_limpio.parquet")
w = wf.loc[wf.fuente_estado.eq("actual") & wf.finca.eq("GAITANA")].copy()
fisica = ["IdSiembra","AnioProyeccion","SemanaProyeccion"]
assert not w.duplicated(fisica).any()
assert w[SEM].notna().all().all()
referencia_wf = w.groupby(SEM).agg(
    webflor_estado_descargado=("ProyeccionFinal","sum"),
    siembras_webflor=("IdSiembra","nunique"),conflicto_webflor=("conflicto_cantidad","max"),
    auditoria_webflor_min=("fecha_auditoria","min"),auditoria_webflor_max=("fecha_auditoria","max")).reset_index()
assert referencia_wf.webflor_estado_descargado.sum()==w.ProyeccionFinal.sum()
antes = len(base)
base = base.merge(referencia_wf,on=SEM,how="left",validate="one_to_one",indicator=True)
reconciliaciones.append({"union":"webflor_actual","cardinalidad":"1:1","antes":antes,"despues":len(base),
 "sin_pareja":int(base._merge.eq("left_only").sum()),"tallos":int(base.tallos_reales.sum())})
base = base.drop(columns="_merge")
base["webflor_origen_certificado"] = False
assert len(base)==antes and base.tallos_reales.sum()==tallos_fuente
display(wf.groupby(["fuente_estado","finca"]).size().to_frame("registros_originales"))
print("Parejas con estado actual:",int(base.webflor_estado_descargado.notna().sum()),"de",len(base))
diagnostico_vigencia = []
for estado,g in wf.loc[wf.finca.eq("GAITANA")].groupby("fuente_estado"):
    diagnostico_vigencia.append({"estado":estado,"filas":len(g),
        "generacion_confirmada":int(g.fecha_generacion_confirmada.notna().sum()),
        "auditoria_posterior_al_objetivo_pct":100*(g.fecha_auditoria.ge(g.semana_inicio+pd.Timedelta(days=7))).mean(),
        "conflictos":int(g.conflicto_cantidad.sum())})
display(pd.DataFrame(diagnostico_vigencia))
print("DECISION: no ejecutar un as-of histórico con fecha_auditoria como si fuera generación.")
print("Si se acreditan emisiones: elegir la última conocida <= origen, por su ámbito y objetivo;")
print("detener vigencia ante sustitución/retirada. No rellenar hacia atrás desde una erradicación.")
del wf, w


registros_originales
fuente_estado             finca                             
actual                    ARABELLA                     98604
                          GAITANA                     209845
archivado                 ARABELLA                    476760
                          GAITANA                     877796
archivado_diario          GAITANA                      22985
captura_previa_sin_origen GAITANA PPTO                    44

Parejas con estado actual: 4056 de 44053


,estado,filas,generacion_confirmada,auditoria_posterior_al_objetivo_pct,conflictos
0,actual,209845,0,27.660893,0
1,archivado,877796,0,78.528041,16
2,archivado_diario,22985,0,100.000000,0


DECISION: no ejecutar un as-of histórico con fecha_auditoria como si fuera generación.
Si se acreditan emisiones: elegir la última conocida <= origen, por su ámbito y objetivo;
detener vigencia ante sustitución/retirada. No rellenar hacia atrás desde una erradicación.


## 7. Validar TODAS las uniones, no sólo mirar el total

Una comparación final 1:1 demuestra que cada llave original tiene la misma cantidad. Se revisan filas sin pareja en cada paso y cobertura de variables; no confundir falta de una variable auxiliar con pérdida de producción.

In [8]:
control = pd.DataFrame(reconciliaciones)
display(control)
assert control.antes.eq(control.despues).all()
assert control.tallos.eq(tallos_fuente).all()
verificacion = referencia_real.merge(base[SEM+["tallos_reales"]],on=SEM,how="outer",validate="one_to_one",suffixes=("_antes","_despues"),indicator=True)
assert verificacion._merge.eq("both").all()
assert verificacion.tallos_reales_antes.eq(verificacion.tallos_reales_despues).all()
assert not base.duplicated(SEM).any()
display(base[["temperatura_semana","plantas_plano","edad_media_plano","potencial_descriptivo",
              "estimado_ingeniero_h1","estimado_ingeniero_h5","webflor_estado_descargado"]].notna().mean().mul(100).round(2).to_frame("cobertura_pct"))
display(base.dtypes.to_frame("tipo"))
print("Conservación exacta de todas las llaves:",len(base),"Total:",int(base.tallos_reales.sum()))

# Explicar ausencias sin perder la fila ni el volumen real.
for variable,referencia in [("temperatura_semana",clima_sem),("plantas_plano",siembra_sem),
                            ("estimado_ingeniero_h1",e.loc[e.horizonte.eq(1)])]:
    faltantes = base.loc[base[variable].isna(),SEM+["tallos_reales"]].copy()
    faltantes["motivo"] = np.where(faltantes.semana_inicio.isin(referencia.semana_inicio),
        "existe semana; dimensiones sin pareja admisible o dato faltante",
        "no hay referencia admisible en esa semana")
    print("AUSENCIAS:",variable)
    display(faltantes.groupby("motivo").agg(filas=("tallos_reales","size"),tallos_conservados=("tallos_reales","sum")))
    display(faltantes.groupby("producto").agg(filas=("tallos_reales","size"),tallos_conservados=("tallos_reales","sum")).sort_values("filas",ascending=False).head(8))


,union,cardinalidad,antes,despues,sin_pareja,tallos
0,clima,N:1,44053,44053,678,327991200
1,planos_curvas,1:1,44053,44053,901,327991200
2,ingeniero_h1,1:1,44053,44053,22195,327991200
3,ingeniero_h2,1:1,44053,44053,22008,327991200
4,ingeniero_h3,1:1,44053,44053,22188,327991200
5,ingeniero_h4,1:1,44053,44053,22355,327991200
6,ingeniero_h5,1:1,44053,44053,22561,327991200
7,webflor_actual,1:1,44053,44053,39997,327991200


,cobertura_pct
temperatura_semana,98.46
plantas_plano,97.95
edad_media_plano,97.20
potencial_descriptivo,20.56
estimado_ingeniero_h1,49.60
estimado_ingeniero_h5,48.77
webflor_estado_descargado,9.21


,tipo
finca,string
producto,string
color,string
variedad,string
semana_inicio,datetime64[ms]
...,...
siembras_webflor,float64
conflicto_webflor,object
auditoria_webflor_min,datetime64[us]
auditoria_webflor_max,datetime64[us]


Conservación exacta de todas las llaves: 44053 Total: 327991200
AUSENCIAS: temperatura_semana


,filas,tallos_conservados
motivo,,
no hay referencia admisible en esa semana,678,6423127


...

,filas,tallos_conservados
motivo,,
existe semana; dimensiones sin pareja admisible o dato faltante,380,1771956
no hay referencia admisible en esa semana,521,4740188


,filas,tallos_conservados
producto,,
CARNATION,344,1114025
MINICARNATION,263,4072900
SOLOMIO,98,606191
BARBATUS,46,55185
RAFFINE,41,374490
STATICE,36,20780
VERONICA,18,29417
GREEN BALL,12,114570


AUSENCIAS: estimado_ingeniero_h1


,filas,tallos_conservados
motivo,,
existe semana; dimensiones sin pareja admisible o dato faltante,7682,2510686
no hay referencia admisible en esa semana,14521,105029156


,filas,tallos_conservados
producto,,
CARNATION,9937,19088004
MINICARNATION,6991,72402051
SOLOMIO,2284,5951551
RAFFINE,854,4736206
BARBATUS,703,769956
VERONICA,473,891098
GREEN BALL,213,2581038
CAMPANULA,188,89066


## 8. Exportar la única base analítica

No se genera otra base_modelos, versión temporal ni manifiesto. Las tablas temporales anteriores sólo viven en memoria. Si falla una validación, no se llega a esta exportación.

In [9]:
base = base.sort_values(SEM).reset_index(drop=True)
base.to_parquet(SALIDA/"base_analitica.parquet",index=False)
print("SALIDA OFICIAL:",SALIDA/"base_analitica.parquet")
print("Filas/columnas:",base.shape)
display(base.head())


SALIDA OFICIAL: C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos_analiticos_proyecto\base_analitica.parquet
Filas/columnas: (44053, 108)


,finca,producto,color,variedad,semana_inicio,tallos_reales,registros_corte,dias_con_reporte,unidad,produccion_observada,anio,semana,temperatura_semana,humedad_semana,radiacion_semana,temperatura_min_semana,temperatura_max_semana,...,archivo_ingeniero_h5,archivo_posterior_ingeniero_h5,desfase_archivo_ingeniero_h5,metodo_union_ingeniero_h5,color_estimado_ingeniero_h5,conflicto_ingeniero_h5,cantidad_incompleta_ingeniero_h5,desviacion_ingeniero_h5,desviacion_pct_ingeniero_h5,equivalencia_exportable_confirmada,fecha_emision_ingeniero_confirmada,webflor_estado_descargado,siembras_webflor,conflicto_webflor,auditoria_webflor_min,auditoria_webflor_max,webflor_origen_certificado
0,GAITANA,ACHILLEA,LIGHT PINK,SASSY SUMMER TAFFY,2023-07-17,60,1,1,tallos reportados,True,2023,29,14.109050,70.429336,126.250440,10.4,19.6,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,<NA>,False,False,NaN,NaN,NaN,NaT,NaT,False
1,GAITANA,ACHILLEA,LIGHT PINK,SASSY SUMMER TAFFY,2023-07-24,185,4,3,tallos reportados,True,2023,30,14.804312,83.266147,173.424899,7.3,21.7,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,<NA>,False,False,NaN,NaN,NaN,NaT,NaT,False
2,GAITANA,ACHILLEA,LIGHT PINK,SASSY SUMMER TAFFY,2023-07-31,180,3,3,tallos reportados,True,2023,31,14.229129,82.855750,169.997847,7.9,21.2,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,<NA>,False,False,NaN,NaN,NaN,NaT,NaT,False
3,GAITANA,ACHILLEA,LIGHT PINK,SASSY SUMMER TAFFY,2023-08-07,120,2,2,tallos reportados,True,2023,32,13.980263,81.903518,177.054872,5.4,22.9,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,<NA>,False,False,NaN,NaN,NaN,NaT,NaT,False
4,GAITANA,ACHILLEA,LIGHT PINK,SASSY SUMMER TAFFY,2023-08-21,60,1,1,tallos reportados,True,2023,34,14.460139,78.082566,219.175714,3.6,22.8,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,<NA>,False,False,NaN,NaN,NaN,NaT,NaT,False


## Resumen: qué está integrado y qué no

- Producción semanal completa del alcance Gaitana: target observado.
- Clima contemporáneo: contexto retrospectivo; para pronosticar se rezaga.
- Plantas/edad del plano de la misma semana y potencial empírico descriptivo; cobertura visible.
- Estimados del ingeniero H1–H5, con origen nominal/archivo y conflictos explícitos.
- Estado actual WebFlor en semanas coincidentes, sin inventar corridas históricas.
- Archivo WebFlor y coeficientes: auxiliares, no duplicados del target ni predictores históricos admitidos.

**Pendientes:** equivalencia reportado/exportable, fecha de emisión real, unidades climáticas, identidad física de estación, publicación del plano/curvas y correspondencias comerciales ambiguas. El siguiente notebook hace EDA; el 08 prepara features y validación, sin escoger un modelo ganador.

## Relación entre las salidas oficiales

```mermaid
flowchart TD
    R["Produccion limpia: movimiento original"]
    P["Base semanal: finca + producto + color + variedad + semana"]
    C["Clima limpio: finca + instante"]
    S["Planos limpios: edicion + linea de cama"]
    U["Curvas empiricas: piloto + siembra + semana de corte"]
    E["Estimados: archivo + dimension + semana objetivo"]
    W["WebFlor: estado o archivo + ID fisico"]
    K["Coeficientes WebFlor: curva + edad"]
    A["base_analitica.parquet: misma produccion, variables adicionales"]
    R -->|"sumar tallos una vez por llave semanal"| P
    C -->|"medicion a dia a semana; N a 1 por finca y semana"| A
    S -->|"edicion elegida; agregar por llave semanal; 1 a 1"| A
    U -->|"producto y variedad univocos; edad; potencial solo descriptivo"| S
    E -->|"un horizonte por columna; conflictos no se suman; 1 a 1"| A
    W -->|"solo estado actual agregado por llave semanal; 1 a 1"| A
    P -->|"tabla izquierda; cero perdida de tallos"| A
    K -.->|"auxiliar: sin vigencia historica certificada"| W
```

Las flechas punteadas son relaciones interpretativas, no joins adicionales ejecutados. El archivo WebFlor y el coeficiente permanecen en sus dos tablas limpias; no se suman al estado actual ni se inventa un origen de pronóstico.
